In [1]:
import re
import cv2
import pytesseract
import pandas as pd
from pathlib import Path

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
processed_dir = Path("../data/processed")
df = pd.read_csv("../data/annotated/ground_truth.csv")
print("Ground truth loaded:")
print(df[["filename", "amount", "date", "condition"]])

Ground truth loaded:
          filename  amount        date condition
0   receipt_01.jpg    50.0  2026-05-10     fresh
1   receipt_02.jpg   203.0  2026-05-10     fresh
2   receipt_03.jpg    44.0  2026-05-10     fresh
3   receipt_04.jpg    24.0  2026-05-11     fresh
4   receipt_05.jpg  1300.0  2026-05-10     fresh
5   receipt_06.jpg  2000.0  2026-05-11     fresh
6   receipt_07.jpg    11.0  2026-05-11     fresh
7   receipt_08.jpg    73.0  2026-05-11     fresh
8   receipt_09.jpg    10.0  2026-05-11     faded
9   receipt_10.jpg    81.0  2026-05-11     faded
10  receipt_11.jpg    59.0  2026-05-10     faded
11  receipt_12.jpg     7.0  2026-05-11     faded
12  receipt_13.jpg    22.0  2026-05-11     faded
13  receipt_14.jpg     8.0  2026-05-10     faded
14  receipt_15.jpg    38.0  2026-05-10  withered
15  receipt_16.jpg    50.0  2026-05-10  withered
16  receipt_17.jpg    59.0  2026-05-10  withered


In [2]:
def extract_amount(text):
    lines = text.split('\n')

    for i, line in enumerate(lines):
        line_clean = line.strip()
        if not line_clean:
            continue

        if re.search(r'\b(CASH|CHANGE|VATABLE|VAT|ZERO|EXEMPT|NET AMT)\b',
                     line_clean, re.IGNORECASE):
            continue

        line_fixed = line_clean
        line_fixed = re.sub(r'[§$£€]', '5', line_fixed)
        line_fixed = re.sub(r'\bAinount\b', 'Amount', line_fixed)
        line_fixed = re.sub(r'\bBue\b', 'Due', line_fixed)
        line_fixed = re.sub(r'cp\)', '(1)', line_fixed)

        is_total = re.search(
            r'(Total\s+Amount\s+Due|Total\s*[\(\[]\d+[\)\]]|TOTAL\s+AMT)',
            line_fixed, re.IGNORECASE
        )
        if is_total:
            numbers = re.findall(r'\d{1,6}[.,]\d{2,3}', line_fixed)
            if numbers:
                raw = numbers[-1].replace(',', '.')
                try:
                    val = round(float(raw), 2)
                    if 1.0 <= val <= 10000.0:
                        return val
                except:
                    pass

            for j in range(i+1, min(i+3, len(lines))):
                next_line = lines[j].strip()
                if re.search(r'\b(CASH|CHANGE)\b', next_line, re.IGNORECASE):
                    break
                next_fixed = re.sub(r'[§$£€]', '5', next_line)
                numbers = re.findall(r'\d{1,6}[.,]\d{2,3}', next_fixed)
                if numbers:
                    try:
                        val = round(float(numbers[0].replace(',', '.')), 2)
                        if 1.0 <= val <= 10000.0:
                            return val
                    except:
                        pass

    return None


def extract_date(text):
    lines = text.split('\n')

    # Primary: transaction date has time attached (HH:MM:SS)
    for line in lines:
        if re.search(r'Date\s+Issued|BIR|PTU|Permit', line, re.IGNORECASE):
            continue
        line_fixed = re.sub(r'[lI]', '1', line)
        match = re.search(
            r'(\d{2}/\d{2}/\d{4})(?:\([A-Za-z]+\))?\s*\d{2}[;:]\d{2}[;:]\d{2}',
            line_fixed
        )
        if match:
            parts = match.group(1).split('/')
            month, day, year = int(parts[0]), int(parts[1]), int(parts[2])
            if 1 <= month <= 12 and 1 <= day <= 31 and 2020 <= year <= 2030:
                # Convert MM/DD/YYYY → YYYY-MM-DD to match ground truth
                return f"{year}-{month:02d}-{day:02d}"

    # Fallback: any MM/DD/YYYY not on a BIR line
    for line in lines:
        if re.search(r'Date\s+Issued|BIR|PTU|Permit', line, re.IGNORECASE):
            continue
        line_fixed = re.sub(r'[lI](?=\d)', '1', line)
        for m in re.finditer(r'(\d{2}/\d{2}/\d{4})', line_fixed):
            parts = m.group(1).split('/')
            month, day, year = int(parts[0]), int(parts[1]), int(parts[2])
            if 1 <= month <= 12 and 1 <= day <= 31 and 2020 <= year <= 2030:
                # Convert MM/DD/YYYY → YYYY-MM-DD to match ground truth
                return f"{year}-{month:02d}-{day:02d}"

    return None


def extract_store(text):
    lines = text.split('\n')
    for i, line in enumerate(lines):
        if re.search(r'7.?ELEV[EUe][NnWwMm]|7.?ELEUEN|7.?ELEWER|7.?E1',
                     line, re.IGNORECASE):
            for j in range(i+1, min(i+6, len(lines))):
                next_line = lines[j].strip()
                if len(next_line) > 5 and re.search(r'[a-zA-Z]{3,}', next_line):
                    return next_line
    return None


def extract_category(text):
    keywords = {
        "food": [
            "sisig", "hotdog", "sandwich", "donut", "bite", "burger",
            "beef", "chicken", "rice", "meal", "bread", "egg", "premasado",
            "taquito", "siopao", "bun", "bm", "goto"
        ],
        "beverage": [
            "water", "juice", "coffee", "drink", "gulp", "soda",
            "tea", "milk", "coke", "pepsi", "energy", "bottle",
            "refresh", "pet", "dw", "mineral", "purified"
        ],
        "personal care": [
            "soap", "shampoo", "tissue", "hygiene",
            "lotion", "deodorant", "toothpaste", "dishpillow", "dishwash"
        ],
        "utilities": [
            "bill", "payment", "7-connect", "gcash", "load",
            "electric", "recharge", "connect", "convenience fee"
        ],
        "snacks": [
            "chips", "candy", "chocolate", "cookie", "popcorn",
            "nuts", "choco", "mucho", "tigerbit", "cracker",
            "lays", "pillows", "chicharon", "jnj", "mr chips"
        ],
    }
    text_lower = text.lower()
    scores = {cat: 0 for cat in keywords}
    for cat, words in keywords.items():
        for word in words:
            if word in text_lower:
                scores[cat] += 1
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "others"

In [3]:
img = cv2.imread(str(processed_dir / "receipt_01.jpg"))
config = "--psm 6 -l eng"
text = pytesseract.image_to_string(img, config=config)

print(f"Raw text:\n{text[:300]}")
print(f"\n--- Extracted Fields ---")
print(f"Amount:   {extract_amount(text)}")
print(f"Date:     {extract_date(text)}")
print(f"Store:    {extract_store(text)}")
print(f"Category: {extract_category(text)}")

Raw text:
° -ELEYE,
Neribrog Ine, .
° Owned ¢ Operated by: Ner tbrog
Ine,
VATREGTIN ¥479-423-988 000
Unit 4 3 9, Rea] Nare Bul ld ing,
: Pinikitan, Cagayan De Org City,
Misamis Orental, Phi pp inigs
Tal 2: (02) 0000009
05/10/2026 (sun) 17:06:05
. INVOICE #1456745 RESET_cNTyo
STORE#2006 SN# 9200064.
MIN gs 210

--- Extracted Fields ---
Amount:   None
Date:     2026-05-10
Store:    None
Category: snacks


In [4]:
extraction_results = []

for _, row in df.iterrows():
    img = cv2.imread(str(processed_dir / row["filename"]))
    if img is None:
        print(f"Could not read {row['filename']}")
        continue

    config = "--psm 6 -l eng"
    text = pytesseract.image_to_string(img, config=config)

    ext_amount   = extract_amount(text)
    ext_date     = extract_date(text)
    ext_store    = extract_store(text)
    ext_category = extract_category(text)

    amount_correct = (ext_amount == row["amount"])
    date_correct   = (ext_date == row["date"])

    extraction_results.append({
        "filename":           row["filename"],
        "condition":          row["condition"],
        "true_amount":        row["amount"],
        "extracted_amount":   ext_amount,
        "true_date":          row["date"],
        "extracted_date":     ext_date,
        "extracted_store":    ext_store,
        "extracted_category": ext_category,
        "amount_correct":     amount_correct,
        "date_correct":       date_correct,
    })

    status_a = "✓" if amount_correct else "✗"
    status_d = "✓" if date_correct else "✗"
    print(f"{row['filename']} | amount: {ext_amount} {status_a} | date: {ext_date} {status_d} | {ext_category}")

results_df = pd.DataFrame(extraction_results)
results_df.to_csv("../outputs/metrics/extraction_results.csv", index=False)
print("\nSaved!")

receipt_01.jpg | amount: None ✗ | date: 2026-05-10 ✓ | snacks
receipt_02.jpg | amount: None ✗ | date: None ✗ | utilities
receipt_03.jpg | amount: 44.0 ✓ | date: 2026-05-10 ✓ | beverage
receipt_04.jpg | amount: 24.0 ✓ | date: 2020-08-01 ✗ | others
receipt_05.jpg | amount: None ✗ | date: None ✗ | utilities
receipt_06.jpg | amount: None ✗ | date: None ✗ | utilities
receipt_07.jpg | amount: None ✗ | date: 2020-08-01 ✗ | others
receipt_08.jpg | amount: None ✗ | date: None ✗ | others
receipt_09.jpg | amount: None ✗ | date: None ✗ | others
receipt_10.jpg | amount: 81.0 ✓ | date: None ✗ | food
receipt_11.jpg | amount: 59.0 ✓ | date: 2026-05-10 ✓ | utilities
receipt_12.jpg | amount: None ✗ | date: 2020-08-01 ✗ | beverage
receipt_13.jpg | amount: 22.0 ✓ | date: 2026-05-11 ✓ | others
receipt_14.jpg | amount: None ✗ | date: None ✗ | beverage
receipt_15.jpg | amount: 38.0 ✓ | date: 2026-05-10 ✓ | others
receipt_16.jpg | amount: 50.0 ✓ | date: 2026-05-10 ✓ | others
receipt_17.jpg | amount: 59.0 ✓ | 

In [5]:
print("=== EXTRACTION SUMMARY ===")
print(f"Total receipts: {len(results_df)}")
print(f"Amount correct: {results_df['amount_correct'].sum()}/{len(results_df)}")
print(f"Date correct:   {results_df['date_correct'].sum()}/{len(results_df)}")
print(f"\nBy condition:")
print(results_df.groupby('condition')[['amount_correct','date_correct']].mean().round(2))
print(f"\nFull results:")
print(results_df[['filename','true_amount','extracted_amount','amount_correct','true_date','extracted_date','date_correct']])

=== EXTRACTION SUMMARY ===
Total receipts: 17
Amount correct: 8/17
Date correct:   7/17

By condition:
           amount_correct  date_correct
condition                              
faded                0.50          0.33
fresh                0.25          0.25
withered             1.00          1.00

Full results:
          filename  true_amount  extracted_amount  amount_correct   true_date  \
0   receipt_01.jpg         50.0               NaN           False  2026-05-10   
1   receipt_02.jpg        203.0               NaN           False  2026-05-10   
2   receipt_03.jpg         44.0              44.0            True  2026-05-10   
3   receipt_04.jpg         24.0              24.0            True  2026-05-11   
4   receipt_05.jpg       1300.0               NaN           False  2026-05-10   
5   receipt_06.jpg       2000.0               NaN           False  2026-05-11   
6   receipt_07.jpg         11.0               NaN           False  2026-05-11   
7   receipt_08.jpg         73.0   

In [6]:
print("=== RAW OCR DEBUG FOR ALL RECEIPTS ===\n")

for _, row in df.iterrows():
    img = cv2.imread(str(processed_dir / row["filename"]))
    if img is None:
        continue

    config = "--psm 6 -l eng"
    text = pytesseract.image_to_string(img, config=config)

    print(f"\n{'='*50}")
    print(f"FILE: {row['filename']} | TRUE AMOUNT: {row['amount']} | TRUE DATE: {row['date']}")
    print(f"{'='*50}")
    print(text[:500])
    print("...")

=== RAW OCR DEBUG FOR ALL RECEIPTS ===


FILE: receipt_01.jpg | TRUE AMOUNT: 50.0 | TRUE DATE: 2026-05-10
° -ELEYE,
Neribrog Ine, .
° Owned ¢ Operated by: Ner tbrog
Ine,
VATREGTIN ¥479-423-988 000
Unit 4 3 9, Rea] Nare Bul ld ing,
: Pinikitan, Cagayan De Org City,
Misamis Orental, Phi pp inigs
Tal 2: (02) 0000009
05/10/2026 (sun) 17:06:05
. INVOICE #1456745 RESET_cNTyo
STORE#2006 SN# 9200064.
MIN gs 2106171638 1259046
[STAFF Apr qy Rose J, Zabal lero
NaturesPur ipwsogm 7 15,00y
nd Me Chips 989 35,00
Tota] (2) 50.00
CASH 50.00
CHANGE 0.00
Vatable 44,64
VAT_Amt 5.36
Zero_Rated Sales 0,00
VAT Exempt Sa
...

FILE: receipt_02.jpg | TRUE AMOUNT: 203.0 | TRUE DATE: 2026-05-10
ELEVEN,
V
/ i Neribros Inc.
L Ohned g Operated by: Heribros
Ine,
uy VATRE@ Try #479-429 968 -0gq
; prit 4&5, Rea] Maro Bul Iding, ‘
Mea cama be Oro City,
ental, Hopj
Ta] Hr ¢ 02)o00aaq9" es
, 95/10/2026, Sun) 16:34,94
. AR #109989, RESE 0
STORE#2006 shi oz5cn (
MIN @, 2106171638 1258246
STAFF Apri] Rose y, Zabal lero
Kk 

In [7]:
for filename in ["receipt_02.jpg", "receipt_05.jpg", "receipt_06.jpg", "receipt_07.jpg"]:
    img = cv2.imread(str(processed_dir / filename))
    config = "--psm 6 -l eng"
    text = pytesseract.image_to_string(img, config=config)
    
    print(f"\n{'='*50}")
    print(f"FILE: {filename}")
    print(f"{'='*50}")
    lines = text.split('\n')
    for i, line in enumerate(lines):
        if re.search(r'total|amount|due|cash|change', line, re.IGNORECASE):
            start = max(0, i-1)
            end = min(len(lines), i+4)
            for l in lines[start:end]:
                print(repr(l))
            print("---")


FILE: receipt_02.jpg
'7-E leven chargas a 1%'
'Cotivenience fee on ay] GCash'
'Cash In transact jang.'
'This transact jan Is non-'
'refundab |. For Inquiries,'
---
'Cotivenience fee on ay] GCash'
'Cash In transact jang.'
'This transact jan Is non-'
'refundab |. For Inquiries,'
'kindly Cal] 2999 for mobi Te or'
---
'(02) 7392880 for landling. |'
'Total ¢ 1) 203.00'
'CASH » 205.00'
'CHANGE 2.00'
'\\'
---
'Total ¢ 1) 203.00'
'CASH » 205.00'
'CHANGE 2.00'
'\\'
'Loyalty Nor my'
---
'CASH » 205.00'
'CHANGE 2.00'
'\\'
'Loyalty Nor my'
'Name; ee'
---

FILE: receipt_05.jpg
'Please bring this slip to the'
': cashier for payment.'
'. Php 1,300.00'
'9653-0027-636)'
'Payment slip valid until |'
---
'May 10, 2026, Sun 3:15 PM'
'GCASH CASH IN'
'Hobile Number ; 09526499364 ,'
'Amount : Phe 1287,00'
'Convenience Fea : Php 13.00 |'
---
'Hobile Number ; 09526499364 ,'
'Amount : Phe 1287,00'
'Convenience Fea : Php 13.00 |'
'For inquiries, kindly call 2882 for'
'mobile or'
---

FILE: receipt_06.jpg
'“2nve